# 03 · Feature Engineering
## Selecció de Gens i Construcció de Features

**Objectiu:** Transformar les dades brutes d'expressió gènica i clíniques en features significatives per al model predictiu, basant-se en coneixement de domini i mètodes computacionals.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('../figures')

print('✅ Mòduls carregats')

## 3.1 Signatures Gèniques basades en Literatura

### Justificació científica:

| Signatura | Origen | Gens | Valor predictiu |
|-----------|--------|------|----------------|
| **IFN-γ 6-gene** | Ayers et al., 2017 (JCI) | CXCL9, CXCL10, IDO1, IFNG, HLA-DRA, STAT1 | Predictiva de resposta pembrolizumab |
| **T cell inflamed GEP** | Ott et al., 2019 | 18 gens | Approvada FDA per pembrolizumab |
| **Immune score** | Rooney et al., 2015 (Cell) | GZMA, PRF1 | Activitat citolítica CTL |
| **TGF-β signature** | Mariathasan et al., 2018 | TGFβ pathway | Resistència en UC |
| **IMPRES** | Auslander et al., 2018 (Nat. Med.) | 15 gens | Resposta anti-PD1 melanoma |

In [ ]:
# ================================================================
# DEFINICIÓ DE SIGNATURES GÈNIQUES
# ================================================================

GENE_SIGNATURES = {
    # Merck IFN-γ 6-gene signature (Ayers et al. 2017, JCI)
    # Validada en múltiples tumors per predir resposta pembrolizumab
    'ifng_6gene': ['CXCL9', 'CXCL10', 'IDO1', 'IFNG', 'HLA-DRA', 'STAT1'],
    
    # T cell inflamed Gene Expression Profile (Ott et al. 2019)
    # 18-gene signature usada en el label pembrolizumab pan-tumor
    'tcell_inflamed': [
        'CXCL9', 'CXCL10', 'IDO1', 'IFNG', 'HLA-DRA', 'STAT1',
        'CD8A', 'GZMB', 'PSMB10', 'LAG3', 'NKG7', 'HLA-E',
        'CMKLR1', 'CXCR6', 'PDCD1LG2', 'TIGIT', 'CD274', 'CCL5'
    ],
    
    # Cytolytic activity (Rooney et al. 2015, Cell)
    'cytolytic': ['GZMA', 'PRF1'],
    
    # Antigen presentation MHC-I
    'mhc_i': ['HLA-A', 'HLA-B', 'HLA-C', 'B2M', 'TAP1', 'TAP2'],
    
    # Immunosuppression signature
    'immunosuppression': ['FOXP3', 'ARG1', 'IDO2', 'CD163', 'IL10'],
    
    # Proliferation (CCLE-based)
    'proliferation': ['MKI67', 'PCNA', 'MCM2'],
}

# Tots els gens únics necessaris
all_sig_genes = list(set([g for sig in GENE_SIGNATURES.values() for g in sig]))
print(f'📊 Signatures definides: {len(GENE_SIGNATURES)}')
print(f'📊 Gens únics totals: {len(all_sig_genes)}')

for name, genes in GENE_SIGNATURES.items():
    print(f'   {name}: {len(genes)} gens')

In [ ]:
# Simular dataset complet (clínic + expressió) per fer feature engineering
n = 68
np.random.seed(42)

# Variables clíniques
response = np.random.binomial(1, 0.45, n)

clinical = pd.DataFrame({
    'patient_id': [f'PT{i:03d}' for i in range(n)],
    'age': np.random.normal(60, 12, n).clip(25, 85),
    'gender': np.random.choice([0, 1], n),  # 0=Female, 1=Male
    'stage_num': np.random.choice([3, 4, 4, 4], n),  # III=3, IV=4
    'ecog_ps': np.random.choice([0, 1, 2], n, p=[0.5, 0.4, 0.1]),
    'prior_therapies': np.random.choice([0, 1, 2, 3], n, p=[0.2, 0.4, 0.3, 0.1]),
    'ldh': np.random.lognormal(5.5, 0.5, n),
    'tmb_raw': np.random.exponential(8, n).clip(0, 80),
    'braf_mut': np.random.choice([0, 1], n, p=[0.58, 0.42]),
    'nras_mut': np.random.choice([0, 1], n, p=[0.72, 0.28]),
    'nf1_mut': np.random.choice([0, 1], n, p=[0.86, 0.14]),
    'b2m_mut': np.random.choice([0, 1], n, p=[0.93, 0.07]),
    'response': response
})

# Matriu d'expressió gènica (log2 TPM+1)
# Gens de totes les signatures
expr_genes = all_sig_genes
n_genes = len(expr_genes)

expr_base = np.random.normal(5, 2, (n, n_genes)).clip(0, 15)

# Patrons biològics
for i, gene in enumerate(expr_genes):
    if gene in GENE_SIGNATURES['ifng_6gene'] + GENE_SIGNATURES['tcell_inflamed']:
        expr_base[response==1, i] += np.random.normal(2, 0.5, response.sum())
    if gene in GENE_SIGNATURES['cytolytic']:
        expr_base[response==1, i] += np.random.normal(1.5, 0.4, response.sum())
    if gene in GENE_SIGNATURES['immunosuppression']:
        expr_base[response==0, i] += np.random.normal(1.5, 0.5, (response==0).sum())
    if gene in GENE_SIGNATURES['mhc_i']:
        expr_base[response==0, i] -= np.random.normal(0.8, 0.3, (response==0).sum())

expr_df = pd.DataFrame(
    expr_base.clip(0, 15),
    columns=expr_genes,
    index=clinical['patient_id']
)

print(f'✅ Dataset generat: {n} pacients')
print(f'   Clínic: {clinical.shape}')
print(f'   Expressió: {expr_df.shape}')

## 3.2 Càlcul de Signatures Gèniques (Scores)

In [ ]:
def compute_signature_score(expr_df, gene_list, method='mean'):
    """
    Calcula un score de signatura gènica a partir de l'expressió.
    
    Methods:
    - 'mean': Mitjana simple de z-scores
    - 'ssgsea': Single-sample GSEA (simplificat)
    - 'sum': Suma de z-scores
    """
    # Filtrar gens disponibles
    available_genes = [g for g in gene_list if g in expr_df.columns]
    missing = [g for g in gene_list if g not in expr_df.columns]
    
    if missing:
        print(f'   ⚠️ Gens no disponibles: {missing}')
    
    if not available_genes:
        return pd.Series(np.nan, index=expr_df.index)
    
    expr_subset = expr_df[available_genes]
    
    if method == 'mean':
        # Z-score per gen, llavors mitjana
        z_scored = (expr_subset - expr_subset.mean()) / (expr_subset.std() + 1e-8)
        return z_scored.mean(axis=1)
    elif method == 'sum':
        z_scored = (expr_subset - expr_subset.mean()) / (expr_subset.std() + 1e-8)
        return z_scored.sum(axis=1)
    else:
        return expr_subset.mean(axis=1)


# Calcular tots els scores de signatures
signature_scores = pd.DataFrame(index=expr_df.index)

for sig_name, genes in GENE_SIGNATURES.items():
    score = compute_signature_score(expr_df, genes, method='mean')
    signature_scores[f'sig_{sig_name}'] = score
    print(f'✅ {sig_name}: score calculat (range: {score.min():.2f} a {score.max():.2f})')

print(f'\n📊 Scores calculats: {signature_scores.shape}')
print(signature_scores.describe())

## 3.3 Features Addicionals Derivades

In [ ]:
# ================================================================
# FEATURE ENGINEERING AVANÇAT
# ================================================================

features = clinical.copy()
features = features.set_index('patient_id')

# 1. SIGNATURES GÈNIQUES (calculades anteriorment)
features = features.join(signature_scores)

# 2. TMB CATEGORIES (basades en thresholds clínics)
# Threshold FDA: TMB-H = 10 mut/Mb (aprovació pan-tumor pembrolizumab)
features['tmb_high'] = (features['tmb_raw'] >= 10).astype(int)
features['tmb_log'] = np.log1p(features['tmb_raw'])  # Transformació log (skewed)

# 3. LDH NORMALITZAT (LDH/ULN - Upper Limit of Normal)
# ULN tipic: 250 U/L
features['ldh_ulnratio'] = features['ldh'] / 250
features['ldh_elevated'] = (features['ldh'] > 250).astype(int)

# 4. INDEX IMMUNE COMBINAT
# Ratio entre resposta immune (IFN-γ) i immunosupressió (ARG1, FOXP3)
if 'sig_ifng_6gene' in features.columns and 'sig_immunosuppression' in features.columns:
    features['immune_index'] = (
        features['sig_ifng_6gene'] - features['sig_immunosuppression']
    )

# 5. T:REG RATIO (CD8+ / Treg proxy)
if 'CD8A' in expr_df.columns and 'FOXP3' in expr_df.columns:
    cd8 = expr_df['CD8A'].values
    foxp3 = expr_df['FOXP3'].values if 'FOXP3' in expr_df.columns else np.ones(n)
    features['cd8_treg_ratio'] = (cd8 + 0.01) / (foxp3 + 0.01)

# 6. DRIVER MUTATION COMPOSITE
# Número total de mutacions drivers
mut_cols = ['braf_mut', 'nras_mut', 'nf1_mut', 'b2m_mut']
features['n_driver_muts'] = features[mut_cols].sum(axis=1)

# 7. ECOG + STAGE COMPOSITE
features['clinical_risk_score'] = features['ecog_ps'] + (features['stage_num'] == 4).astype(int)

# 8. AGE GROUPS
features['age_group'] = pd.cut(features['age'], bins=[0, 50, 65, 100],
                                 labels=[0, 1, 2]).astype(int)

# Eliminar variables raw que han estat transformades
features = features.drop(columns=['ldh', 'tmb_raw'], errors='ignore')

print(f'✅ Features finals: {features.shape}')
print('\nLlista de features:')
feature_cols = [c for c in features.columns if c != 'response']
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

## 3.4 Selecció de Features — Mètodes Estadístics i ML

In [ ]:
X = features[feature_cols].fillna(features[feature_cols].median())
y = features['response'].values

# ---- Mètode 1: ANOVA F-test ----
selector_f = SelectKBest(f_classif, k='all')
selector_f.fit(X, y)
f_scores = pd.DataFrame({
    'feature': feature_cols,
    'f_score': selector_f.scores_,
    'p_value': selector_f.pvalues_
}).sort_values('f_score', ascending=False)

# ---- Mètode 2: Mutual Information ----
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({
    'feature': feature_cols,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

# ---- Mètode 3: Random Forest Feature Importance ----
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X, y)
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'rf_importance': rf.feature_importances_
}).sort_values('rf_importance', ascending=False)

# ---- Combinar rankings ----
# Rank combinat (Borda count)
f_rank = f_scores.reset_index(drop=True).reset_index().rename(columns={'index': 'f_rank'})
mi_rank = mi_df.reset_index(drop=True).reset_index().rename(columns={'index': 'mi_rank'})
rf_rank = rf_importance.reset_index(drop=True).reset_index().rename(columns={'index': 'rf_rank'})

combined = pd.merge(f_rank[['feature', 'f_rank', 'f_score', 'p_value']], 
                     mi_rank[['feature', 'mi_rank', 'mi_score']], on='feature')
combined = pd.merge(combined, rf_rank[['feature', 'rf_rank', 'rf_importance']], on='feature')
combined['combined_rank'] = combined[['f_rank', 'mi_rank', 'rf_rank']].mean(axis=1)
combined = combined.sort_values('combined_rank')

print('🏆 TOP 15 Features per Ranking Combinat (ANOVA + MI + RF):')
print(combined.head(15)[['feature', 'f_score', 'mi_score', 'rf_importance', 'combined_rank']]
      .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

In [ ]:
# Visualització del ranking de features
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

top_n = 15
palette = sns.color_palette('viridis', top_n)

# F-score
ax = axes[0]
top_f = f_scores.head(top_n)
colors = ['#E74C3C' if p < 0.05 else '#95A5A6' for p in top_f['p_value']]
ax.barh(range(len(top_f)), top_f['f_score'], color=colors)
ax.set_yticks(range(len(top_f)))
ax.set_yticklabels(top_f['feature'], fontsize=9)
ax.set_xlabel('F-score')
ax.set_title('ANOVA F-Test', fontweight='bold')
ax.invert_yaxis()

# Mutual Information
ax = axes[1]
top_mi = mi_df.head(top_n)
ax.barh(range(len(top_mi)), top_mi['mi_score'], color='#3498DB')
ax.set_yticks(range(len(top_mi)))
ax.set_yticklabels(top_mi['feature'], fontsize=9)
ax.set_xlabel('MI Score')
ax.set_title('Mutual Information', fontweight='bold')
ax.invert_yaxis()

# Random Forest
ax = axes[2]
top_rf = rf_importance.head(top_n)
ax.barh(range(len(top_rf)), top_rf['rf_importance'], color='#27AE60')
ax.set_yticks(range(len(top_rf)))
ax.set_yticklabels(top_rf['feature'], fontsize=9)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Importance', fontweight='bold')
ax.invert_yaxis()

plt.suptitle('Selecció de Features — Comparació de Mètodes', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_selection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SELECCIÓ FINAL DE FEATURES
# Combinant ranking estadístic + coneixement de domini

SELECTED_FEATURES = [
    # Signatures gèniques (alta capacitat predictiva + base biològica)
    'sig_ifng_6gene',          # Merck signature - validada en assaigs clínics
    'sig_tcell_inflamed',      # 18-gene GEP - label FDA pembrolizumab
    'sig_cytolytic',           # Activitat citotòxica CTL
    'sig_mhc_i',               # Presentació antigen (mecanisme resistència)
    'sig_immunosuppression',   # Immunosupressió tumoral
    'immune_index',            # Índex combinat immune/supressor
    
    # TMB
    'tmb_log',                 # Càrrega mutacional (transformada log)
    'tmb_high',                # Flag binari FDA threshold
    
    # Clíniques
    'age',
    'gender',
    'ecog_ps',
    'stage_num',
    'prior_therapies',
    'ldh_ulnratio',
    'ldh_elevated',
    'clinical_risk_score',
    
    # Mutacions
    'braf_mut',
    'b2m_mut',                 # Mecanisme resistència conegut
    'n_driver_muts',
]

# Verificar que totes les features estan disponibles
available = [f for f in SELECTED_FEATURES if f in features.columns]
missing = [f for f in SELECTED_FEATURES if f not in features.columns]

if missing:
    print(f'⚠️  Features no disponibles: {missing}')
    SELECTED_FEATURES = available

print(f'✅ Features seleccionades: {len(SELECTED_FEATURES)}')

# Guardar dataset final
final_dataset = features[SELECTED_FEATURES + ['response']].fillna(features[SELECTED_FEATURES].median())
final_dataset.to_csv(PROCESSED_DIR / 'features_final.csv')

print(f'📁 Dataset guardat: {PROCESSED_DIR}/features_final.csv')
print(f'   Shape: {final_dataset.shape}')
print(f'   Distribució resposta: {final_dataset["response"].value_counts().to_dict()}')